# **Cyclist Bike Usage Analysis**
## Data Loading 

In [1]:
# Data Handling Libraries 
import pandas as pd
import numpy as np
import os

# Visualization Libraries 
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots appear inside the notebook 
%matplotlib inline

#Set teh theme of the seaborn plot 
sns.set(style="white") 

In [2]:
path = r"C:\Users\DELL\Documents\GitHub\cyclist_bike_usage_analysis\raw_data" 

files = [f for f in os.listdir(path) if f.endswith(".csv")]
print(files)

['202501_tripdata.csv', '202502_tripdata.csv', '202503_tripdata.csv', '202504_tripdata.csv', '202505_tripdata.csv', '202506_tripdata.csv', '202507_tripdata.csv', '202508_tripdata.csv', '202509_tripdata.csv', '202510_tripdata.csv', '202511_tripdata.csv', '202512_tripdata.csv']


In [ ]:
# Load all the files at once
df_list = [pd.read_csv(os.path.join(path, f)) for f in files] 
main_df = pd.concat(df_list, ignore_index = True) # ignore_index to reset the rows number 

# To know the nuber of rows and columns my dataset has after i have now commbined 
main_df.shape()
main_df.head()

(5552994, 13)

In [ ]:
# check the columns in the dataset 
main_df.columns 

# check data types of eaach column in the dataset 
main_df.dtypes

main_df.isnull().sum() # coming back to it to remove null values 

ride_id                     0
rideable_type               0
started_at                  0
ended_at                    0
start_station_name    1184673
start_station_id      1184673
end_station_name      1243305
end_station_id        1243305
start_lat                   0
start_lng                   0
end_lat                  5535
end_lng                  5535
member_casual               0
dtype: int64

## Data Cleaning

In [ ]:
# dropping null values found in the columns start_station_id and end_station_id 
main_df.dropna(subset=["start_station_id", "end_station_id"], inplace = True)

main_df.isnull().sum()

In [ ]:
# converting the started_at and ended_at column to datetime data types 
main_df["started_at"] = pd.to_datetime(main_df["started_at"])
main_df["ended_at"] = pd.to_datetime(main_df["ended_at"])

# converting the above to minutes that is stored in the ride_length_minutes column
main_df["ride_length"] = main_df["ended_at"] - main_df["started_at"]
main_df["ride_length_minutes"] = (main_df["ride_length"].dt.total_seconds() / 60).round(2)


main_df[["started_at","ended_at","ride_length","ride_length_minutes"]].head()

,started_at,ended_at,ride_length,ride_length_minutes
0,2025-01-21 17:23:54.538,2025-01-21 17:37:52.015,0 days 00:13:57.477000,13.96
1,2025-01-11 15:44:06.795,2025-01-11 15:49:11.139,0 days 00:05:04.344000,5.07
2,2025-01-02 15:16:27.730,2025-01-02 15:28:03.230,0 days 00:11:35.500000,11.59
3,2025-01-23 08:49:05.814,2025-01-23 08:52:40.047,0 days 00:03:34.233000,3.57
4,2025-01-16 08:38:32.338,2025-01-16 08:41:06.767,0 days 00:02:34.429000,2.57


In [ ]:
main_df.pop("ride_length_seconds")

In [ ]:
main_df.duplicated().sum() #Checking for duplicates 

np.int64(0)

In [ ]:
# Checking for outliers in the length of the rides
main_df = main_df[main_df["ride_length_minutes"] <= 120]
main_df["ride_length_minutes"].max()

np.float64(120.0)

In [ ]:
# getting statistical description of the columns 
# "start_lat","end_lat", "start_lng", "end_lng"
main_df[["start_lat","end_lat", "start_lng", "end_lng"]].describe()

# defining the normal longitude and latitude values 
lat_min, lat_max = 41.65, 42.06
lng_min, lng_max = -87.84, -87.75


# condition to keep the rows where the latitude and longitude values
#  meet the conditions below
main_df = main_df[~(
    (main_df["start_lat"] < lat_min) | (main_df["start_lat"] > lat_max) |
    (main_df["end_lat"] < lat_min) | (main_df["end_lat"] > lat_max) |
    (main_df["start_lng"] < lng_min) | (main_df["start_lng"] > lng_max) |
    (main_df["end_lng"] < lng_min) | (main_df["end_lng"] > lng_max)
)]

main_df[["start_lat", "end_lat", "start_lng","end_lng"]].head(30)

## Exploratory Data Analysis 

In [ ]:
main_df.head()

In [ ]:
# Filtering to only people that rode the bike for at least 2 minutes 
main_df = main_df[main_df["ride_length_minutes"] >= 2]

In [ ]:
casual_df = main_df[main_df["member_casual"] == "casual"]
member_df = main_df[main_df["member_casual"] == "member"]

In [ ]:
main_df.columns

### Who uses the bikes more, Casual riders or Member riders

In [95]:
# number of casual riders in the dataset 
total_member_riders = len(member_df)
total_casual_riders = len(casual_df)

In [104]:
# number of memebr riders in percentage = 55.04%
total_riders = len(main_df)
casual_riders_pct = round((total_casual_riders / total_riders * 100), 2)
casual_riders_pct

55.04

In [103]:
# number of memebr riders in percentage = 44.96%
member_riders_pct = round((total_member_riders / total_riders * 100), 2)
member_riders_pct

44.96

### Average length of casual riders bike riding 

In [ ]:
casual_avg_ride = round((casual_df["ride_length_minutes"].mean()),2)
casual_avg_ride

### Average length of member riders bike riding 

In [ ]:
member_avg_ride = round((member_df["ride_length_minutes"].mean()),2)
member_avg_ride

### Most popular day that casual riders ride their bikes 

In [110]:
day_casual = casual_df["started_at"].dt.day_name()
day_member = member_df["started_at"].dt.day_name()

In [ ]:
most_rides_day_casual = day_casual.value_counts()
most_rides_day_member= day_member.value_counts()

most_rides_day_member.head(7)
most_rides_day_casual.head(7)

### What time of the day do casual riders ride their bike the most 

In [ ]:
casual_df["hour_of_day"] = casual_df["started_at"].dt.hour
member_df["hour_of_day"] = member_df["started_at"].dt.hour

rides_per_hour_casual = casual_df["hour_of_day"].value_counts()
rides_per_hour_member = member_df["hour_of_day"].value_counts()

rides_per_hour_member.head()

### What type of bike do Casual riders prefer 

In [124]:
preferred_bike_casual = round((casual_df["rideable_type"].value_counts(normalize = True) * 100), 2)
preferred_bike_casual

rideable_type
electric_bike    56.67
classic_bike     43.33
Name: proportion, dtype: float64

In [125]:
preferred_bike_member = round((member_df["rideable_type"].value_counts(normalize = True) * 100), 2)
preferred_bike_member

rideable_type
electric_bike    54.59
classic_bike     45.41
Name: proportion, dtype: float64

### Which station has the most casual riders 

In [ ]:
preferred_station_casual = casual_df["start_station_name"].value_counts()
preferred_station_casual.head()

## Converting the dataset back to csv to enable it to be exported into power bi 

In [128]:
main_df.to_csv("cyclist_final.csv",index = False)

## Insights 1 - Percentage of casual riders to member riders 

 Casual riders make up approximately 55.04% of total rides while memebr riders make up 44.96% of total riders. This indicates a large user base that is not yet converted into annual memberships. This presents a significant opportunity for targeted conversion strategies

 ## Insights 2 - Average ride duration

 Casual riders have an average ride duration of 17.64 minutes, compared to 12.9 minutes for members. This indicates that casual riders tend to take longer, more leisure-oriented trips, while members likely use bikes for shorter, more routine purposes such as commuting.

 ## Insight 3 – Most Popular Days for Casual Riders vs Member Riders

* **Casual Riders:** The majority of rides occur on Saturday (964) and Sunday (900), with Friday (892) also high. Weekday rides are lower (Monday: 810, Tuesday: 772, Wednesday: 785, Thursday: 776). This indicates casual riders primarily ride for leisure or recreation, not commuting.

* **Member Riders:** Patterns contrast with casual riders. Most rides happen midweek (Wednesday: 806, Tuesday: 769, Thursday: 753 ) while weekend usage is lower (Saturday: 636, Sunday: 510), suggesting routine, commute-focused behavior.

* **Business Implication:** Casual riders’ weekend concentration presents an opportunity. Promotions, incentives, or membership campaigns could be timed around weekends or Friday evenings to encourage casual riders to become full members.


## Insight 4 – Most Popular Time of Day for Casual vs Member Riders

**Casual Riders:**

Peak riding hours are 6 PM (18:00, 468 rides), 7 PM (19:00, 411 rides), and 3–5 PM (15:00–17:00, 409–392 rides).

This suggests casual riders prefer late afternoon and evening rides, likely after work or school hours, reinforcing the idea that they ride primarily for leisure or recreation rather than commuting.

**Member Riders:**

Member riders show a peak in 4–6 PM (16:00–18:00, 423–452 rides), with some activity at 7 PM (19:00, 348 rides) and 3 PM (15:00, 302 rides).

This pattern indicates members are more routine-oriented, likely commuting home from work or school during typical rush hours.

**Business Implication:**

Marketing campaigns or membership promotions targeting casual riders could be timed in the late afternoon and evening, especially around 6–7 PM, when most casual riders are active.

Understanding these peak hours allows the team to maximize engagement with casual riders and offer incentives to encourage conversion to full membership

## Insight 5- Bike Preference of Casual Riders 

**Electric Bikes are most popular:** 56.67% of casual riders choose electric bikes.

**Classic Bikes are less popular:** 43.33% of casual riders use classic bikes.

**Business Implication:** Marketing campaigns or membership perks could highlight electric bike convenience, since the majority of casual riders prefer them. Offering trials, discounts, or loyalty points for electric bike rides could help convert casual riders into full members.


## Insight 5 – Popular Stations for Casual Riders

**Top Stations:**
Casual riders predominantly start their rides at the following stations:

* Milwaukee Ave & Ainslie St: 342 rides
* Narragansett Ave & Wrightwood Ave: 230 rides
* Portage Park: 218 rides
* Austin Blvd & Lake St: 158 rides
* Shabbona Park: 156 rides

**Behavior Interpretation:**

Casual riders are concentrated around specific stations, likely near recreational areas or easily accessible neighborhoods.

These stations represent high-traffic points for casual users, making them ideal locations for targeted promotions, advertisements, or membership campaigns.

**Business Implication:**

By focusing marketing efforts at these stations, such as offering trial membership perks, discounts, or special events, stakeholders can increase the likelihood of converting casual riders to full members.

This aligns with the overall goal of driving membership growth by leveraging behavior patterns from casual riders.